# Avatar thật cho staging với 0đ (Colab T4 + HLS)

Cell dưới dựng **toàn bộ** chuỗi avatar trên một máy Colab, không Docker:

`LiveTalking (rtcpush) → MediaMTX (HLS) → cloudflared (HTTPS) → AI service (FastAPI) → cloudflared (HTTPS)`

Lý do không dùng WebRTC thuần của LiveTalking: kênh media là UDP đi thẳng từ máy chủ tới trình duyệt,
Colab và tunnel HTTP (ngrok/cloudflared) không chuyển được UDP. Xem `docs/AVATAR-HLS-FREE.md`.

**Điều kiện**
- Runtime → Change runtime type → **Hardware accelerator: T4 GPU** (free).
- Lần chạy đầu ~20–30 phút: tải 300MB weights + build avatar pack. Lần sau nhanh hơn nếu chưa mất disk.
- Không đóng tab; cell keepalive bên dưới giữ session (Colab free cắt sau ~12h).

In [ ]:
# Repo AI standalone public nên Colab clone được không cần token.
AI_BRANCH = "dev"
import os
!nvidia-smi -L
if os.path.isdir('/content/ai/.git'):
    # Lần chạy lại: giữ nguyên weights/avatar pack đã tải, chỉ kéo code mới.
    !git -C /content/ai fetch -q origin {AI_BRANCH}
    !git -C /content/ai checkout -q -B {AI_BRANCH} origin/{AI_BRANCH}
else:
    !git clone -q -b {AI_BRANCH} https://github.com/PNV-do-an/AI-Interview-Platform-AI.git /content/ai
# vendored/LiveTalking không vào git, colab_avatar.sh tự clone lipku/LiveTalking
# (pin commit đã kiểm chứng) ở bước 0/8.
!ls -l /content/ai/scripts/colab_avatar.sh

In [ ]:
# Cài deps -> tải weights -> build avatar pack -> MediaMTX -> LiveTalking rtcpush -> tunnel HLS -> AI service -> tunnel AI.
# Chạy nền rồi tail log: !bash giữ pipe của kernel, nên cell treo ở bước 7/8 ngay cả khi tunnel đã OK.
import os, subprocess, time
LOG = '/content/logs/bringup.log'
os.makedirs('/content/logs', exist_ok=True)
open(LOG, 'wb').close()
env = dict(os.environ, AI_ROOT='/content/ai', FE_ORIGIN='https://aiinterview-vn.vercel.app')
proc = subprocess.Popen(['bash', '/content/ai/scripts/colab_avatar.sh'], cwd='/content/ai', env=env,
                      stdin=subprocess.DEVNULL, stdout=open(LOG, 'ab'), stderr=subprocess.STDOUT,
                      start_new_session=True)
print('bringup chạy nền, pid =', proc.pid, '| log =', LOG, flush=True)
pos, waited = 0, 0
while waited < 3000:
    time.sleep(5)
    waited += 5
    size = os.path.getsize(LOG)
    if size > pos:
        with open(LOG, 'rb') as fh:
            fh.seek(pos)
            chunk = fh.read(size - pos)
        pos = size
        print(chunk.decode('utf-8', 'replace'), end='', flush=True)
    if proc.poll() is not None:
        print('--- bringup kết thúc, exit code', proc.returncode, '---', flush=True)
        break

In [ ]:
# In hai URL và xác nhận AI service tự khai báo luồng avatar qua /health.
!cat /content/avatar_urls.txt
!curl -s $(cat /content/ai_url.txt)/health | python3 -m json.tool

In [ ]:
# Giữ session Colab: chạy cell này ở cuối và để nguyên (thread nền ping mỗi 9 phút).
import threading, time

def _keep():
    from google.colab import output
    while True:
        try:
            output.eval_js('new Date().toString()')
        except Exception:
            pass
        time.sleep(9 * 60)

threading.Thread(target=_keep, daemon=True).start()
print('keepalive đang chạy nền (ping mỗi 9 phút)')

## Nối vào staging

1. Mở https://aiinterview-vn.vercel.app/mock-interviews, chọn **LiveTalking Avatar**. Dán URL AI từ `/content/ai_url.txt`, bấm **Kiểm tra & lưu**, rồi vào phòng. URL được lưu trên trình duyệt nên tunnel mới không cần redeploy frontend.\n2. Trong phòng, footer hiện `• Giọng đọc: avatar LiveTalking` và khung video có hình là đã kết nối.\n3. Bấm **Bật tiếng avatar** một lần (Chrome chặn tự phát có tiếng cho tới khi người dùng tương tác).

## Khi phòng tự chuyển về giọng đọc trình duyệt

| Chữ hiện trong phòng | Nghĩa | Kiểm tra |
| --- | --- | --- |
| `Máy chủ avatar không phản hồi` | AI service gọi LiveTalking không được | `tail /content/logs/livetalking.log` |
| `Luồng video avatar bị đứt (...)` | HLS lỗi giữa phiên (Colab mất/tunnel đổi) | `tail /content/logs/mediamtx.out`, chạy lại cell 3 |
| `AI service không phản hồi` | tunnel AI chết | `curl $(cat /content/ai_url.txt)/health` |

Log: `/content/logs/{livetalking,ai,cf-ai,cf-hls}.log` và `/content/logs/mediamtx.out`.